In [3]:
# importar librerías
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [14]:
query = """
SELECT COUNT(*) 
FROM books
WHERE publication_date > '2000-01-01';
"""

pd.io.sql.read_sql(query, con=engine)

,count
0,819


In [16]:
query = """
SELECT COUNT(*) 
FROM reviews
"""
pd.io.sql.read_sql(query, con=engine)


,count
0,2793


In [19]:
query = """
SELECT title, AVG(rating) 
FROM  ratings
INNER JOIN books
ON ratings.book_id = books.book_id
GROUP BY title

"""
pd.io.sql.read_sql(query, con=engine)

,title,avg
0,The Count of Monte Cristo,4.217391
1,Count Zero (Sprawl #2),2.500000
2,The Botany of Desire: A Plant's-Eye View of th...,3.500000
3,The Poisonwood Bible,4.363636
4,The Canterbury Tales,3.333333
...,...,...
994,Of Love and Other Demons,4.500000
995,In the Heart of the Sea: The Tragedy of the Wh...,3.333333
996,Welcome to Temptation (Dempseys #1),5.000000
997,World's End (The Sandman #8),4.500000


In [31]:
query = """
SELECT publisher, COUNT(book_id) AS num_books
FROM  publishers
INNER JOIN books
ON publishers.publisher_id = books.publisher_id
WHERE num_pages > 50
GROUP BY publisher
ORDER BY num_books DESC
LIMIT 1
"""
pd.io.sql.read_sql(query, con=engine)

,publisher,num_books
0,Penguin Books,42


In [30]:
query = """SELECT a.author, AVG(r.rating) AS avg_rating
           FROM authors a
           INNER JOIN books b
           ON a.author_id = b.author_id
           INNER JOIN ratings r
           ON b.book_id = r.book_id
           WHERE b.book_id IN (
                 SELECT book_id
                 FROM ratings
                 GROUP BY book_id
                 HAVING COUNT(*) >= 50
           )
           GROUP BY a.author
           ORDER BY avg_rating DESC
           LIMIT 1
"""
pd.io.sql.read_sql(query, con=engine)


,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.287097


In [32]:
query = """
SELECT 
    AVG(num_reviews) AS avg_reviews
FROM (
    SELECT 
        r.username,
        COUNT(DISTINCT rev.review_id) AS num_reviews
    FROM ratings r
    LEFT JOIN reviews rev
        ON r.username = rev.username
    GROUP BY r.username
    HAVING COUNT(DISTINCT r.book_id) > 50
) t;
"""

pd.io.sql.read_sql(query, con=engine)

,avg_reviews
0,24.333333
